In [8]:
import pandas as pd
cluster_labels = pd.read_csv("../data/processed/cluster_labels.csv")
cluster_labels = cluster_labels["Cluster"]
cluster_labels

0       1
1       1
2       1
3       2
4       1
       ..
7038    2
7039    0
7040    0
7041    0
7042    2
Name: Cluster, Length: 7043, dtype: int64

In [9]:
X_train = pd.read_csv("../data/processed/X_train.csv")
X_test = pd.read_csv("../data/processed/X_test.csv")
# len(X_train) + len(X_test)
y_train = pd.read_csv("../data/processed/y_train.csv").squeeze()
y_test = pd.read_csv("../data/processed/y_test.csv").squeeze()
len(y_test) + len(y_train)

7043

In [10]:

X_train_cluster = X_train.copy()
X_test_cluster = X_test.copy()
n_train = len(X_train)
X_train_cluster["Cluster"] = cluster_labels[:n_train]
X_test_cluster["Cluster"] = cluster_labels[n_train:]


In [11]:
X_train_cluster = pd.get_dummies(
    X_train_cluster,
    columns=["Cluster"],
    dtype=int
)

X_test_cluster = pd.get_dummies(
    X_test_cluster,
    columns=["Cluster"],
    dtype=int
)

# Use the training feature schema for the test set.
X_test_cluster = X_test_cluster.reindex(
    columns=X_train_cluster.columns,
    fill_value=0,
)

print(X_train_cluster.filter(regex="^Cluster_").head())

   Cluster_0  Cluster_1  Cluster_2
0          0          1          0
1          0          1          0
2          0          1          0
3          0          0          1
4          0          1          0


### training top models from phase 4 with the cluster feature 

top_models = [
    "Logistic Regression",
    "SVM",
    "XGBoost"
]


In [12]:
from sklearn.base import clone
from sklearn.linear_model import LogisticRegression
from sklearn.svm import SVC
from xgboost import XGBClassifier


negative_count = (y_train == 0).sum()
positive_count = (y_train == 1).sum()

scale_pos_weight = float(negative_count/ positive_count)

best_models = {
    "Logistic Regression": LogisticRegression(
        random_state=42,
        max_iter=1000,
        class_weight="balanced"
    ),
    "SVM": SVC(
        random_state=42,
        class_weight="balanced"
    ),
    "XGBoost": XGBClassifier(
        random_state=42,
        eval_metric="logloss",
        scale_pos_weight=scale_pos_weight
    )
}

trained_cluster_models = {}
for name, model in best_models.items():
    model.fit(X_train_cluster, y_train)
    trained_cluster_models[name] = model
print("retrained all models with the cluster feature")

retrained all models with the cluster feature


In [13]:
from sklearn.base import clone
from sklearn.calibration import CalibratedClassifierCV
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    roc_auc_score,
)

# Top models selected in 04_classification.ipynb (without the cluster feature).
top_models = [
    "Logistic Regression",
    "SVM",
    "XGBoost",
]


def evaluate_model(model, X_fit, X_eval):
    model = clone(model)
    if isinstance(model, SVC):
        model = CalibratedClassifierCV(estimator=model, cv=5, ensemble=False)

    model.fit(X_fit, y_train)
    y_pred = model.predict(X_eval)
    y_proba = model.predict_proba(X_eval)[:, 1]

    return {
        "accuracy": accuracy_score(y_test, y_pred),
        "precision": precision_score(y_test, y_pred, zero_division=0),
        "recall": recall_score(y_test, y_pred, zero_division=0),
        "f1": f1_score(y_test, y_pred, zero_division=0),
        "roc_auc": roc_auc_score(y_test, y_proba),
    }


comparison = []
for name in top_models:
    baseline_metrics = evaluate_model(best_models[name], X_train, X_test)
    cluster_metrics = evaluate_model(
        best_models[name], X_train_cluster, X_test_cluster
    )

    for feature_set, metrics in [
        ("Without cluster", baseline_metrics),
        ("With cluster", cluster_metrics),
    ]:
        comparison.append({
            "model": name,
            "features": feature_set,
            **metrics,
        })

comparison_df = pd.DataFrame(comparison).set_index(["model", "features"])
comparison_df.round(4)


accuracy  precision  recall      f1  \
model               features                                               
Logistic Regression Without cluster    0.7410     0.5078  0.7834  0.6162   
                    With cluster       0.7410     0.5077  0.7888  0.6178   
SVM                 Without cluster    0.7899     0.6005  0.6230  0.6115   
                    With cluster       0.7899     0.6226  0.5294  0.5723   
XGBoost             Without cluster    0.7736     0.5621  0.6658  0.6095   
                    With cluster       0.7665     0.5623  0.5428  0.5524   

                                     roc_auc  
model               features                  
Logistic Regression Without cluster   0.8418  
                    With cluster      0.8406  
SVM                 Without cluster   0.8229  
                    With cluster      0.8241  
XGBoost             Without cluster   0.8251  
                    With cluster      0.8129

In [14]:
comparison_df.round(4)

accuracy  precision  recall      f1  \
model               features                                               
Logistic Regression Without cluster    0.7410     0.5078  0.7834  0.6162   
                    With cluster       0.7410     0.5077  0.7888  0.6178   
SVM                 Without cluster    0.7899     0.6005  0.6230  0.6115   
                    With cluster       0.7899     0.6226  0.5294  0.5723   
XGBoost             Without cluster    0.7736     0.5621  0.6658  0.6095   
                    With cluster       0.7665     0.5623  0.5428  0.5524   

                                     roc_auc  
model               features                  
Logistic Regression Without cluster   0.8418  
                    With cluster      0.8406  
SVM                 Without cluster   0.8229  
                    With cluster      0.8241  
XGBoost             Without cluster   0.8251  
                    With cluster      0.8129